# Datamine PICDIR Process Example

This notebook demonstrates how to configure and run the **`picdir`** process wrapper in `dmstudio`.

## Process Description

## PICDIR

Writes file names to an output file (catalogue file) if the file name or field names within the file match the given pattern expressions.

### Expression Syntax

In the following text all keywords are capitalized. When running **PICDIR** , any keyword may be entered in either uppercase or lowercase and abbreviated to its first three (or more) characters.

The syntax of a pattern matching expression is:

|  [FILE] |  [REGEXP] |  <pattern> |
---|---|---|---|---
or |  FIELD |  [REGEXP] |  <pattern> |
or |  FIELD |  <kind> |  |
or |  FIELD |  [REGEXP] |  <pattern> , |  <kind>

The keyword "FILE" is optional. If a pattern is to be matched against field names, the keyword "FIELD" must be included. The field "kind" will be matched against the type (numeric or alphanumeric) and storage class (explicit or implicit) of the fields in any file.

The keyword "REGEXP" is normally omitted, in which case "pattern" may consist of literal characters to be matched, or one of the following elements:

? |  Any single character
---|---
* |  Wildcard. A group of zero or more characters
[...] |  Any one of the characters enclosed in the square brackets. The short hand notation "a-z" means any lowercase letter; refer to the examples below for more details.
[^...] |  Any character except one of these

The special meaning of a character (e.g. "*") is lost if the character is preceded by "\", hence the match a literal "*", use "\\*". Quotes (double or single) may be used to enclose patterns if desired.

Where a pattern is applied to file names, lower case letters are translated into upper case before the pattern is used.

If the keyword "REGEXP" is used, the pattern specifies a full regular expression. Regular expressions allow advanced users to make more complex selections than are possible by using the pattern elements specified above.

A regular expression in **PICDIR** may contain the following elements.

% |  Matches the beginning of the file or field name
---|---
$ |  Matches the end of the file or field name
* |  Zero or more occurrences of the preceding pattern element.
? |  As above
[...] |  As above
[^...] |  As above

The "kind" is a list of field type or storage class specifiers, separated by commas. Keywords that specify field type are "NUMERIC" or "ALPHANUMERIC". Storage classes are "EXPLICIT" or "IMPLICIT". Of course, any of these may be abbreviated to three (or more) letters.

If more than one type or class keyword is given, the last one specified is used.

### Concatenation of Expressions

The result of a pattern matching expression is either TRUE or FALSE.

Any result may be inverted by preceding the expression by the keyword "NOT" (for example, "NOT XPICRT,IMPLICIT"). Two expressions may be joined together by "AND" or "OR" operators. The result is another expression.

The "AND" operator has higher precedence than "OR". Parentheses (brackets) may be used to override the normal order of evaluation of expressions.

### Input Files:

### Output Files:

* **out** (Catalogue):
  Output catalogue file, giving list of files.
  Required=Yes

### Fields:

* **file** (Character : OUT):
  Optional name for the field that is to contain the file names. The default is
  "'FILENAM", i.e. PICDIR will produce a catalogue file.
  Default=FILENAM
  Required=No

### Parameters:

* **append**:
  If set to 1 then selected field names will be appended to the **OUT** file, provided it
  exists and has a valid DD (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **print**:

* **Options** (0: No display of matching file names >0 Display file names as they are):
  selected. (0)
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **sort**:
  If set to 1 then the output file will be sorted after all file names have been written
  to it (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **longname**:
  If set to 1 then the fields **LOGICAL** (5A4) and **SYSTEM** (32A4) will be added to
  the output file. **LOGICAL** is the full, logical (long) name of the file. **SYSTEM**
  contains the full path name of the file. The default for **LONGNAME** is (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('picdir')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute picdir
print("Running picdir...")
dm_fil.picdir(
    out_o='t_picdir_out',  # required
    # file_f='optional',  # optional
    # append_p=0,  # optional
    # print_p=0,  # optional
    # sort_p=0,  # optional
    # longname_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("picdir execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_picdir_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")